# M4 · Componente visual: `ver()`  (Camilo)

Notebook para **Colab con GPU** (Entorno de ejecución > Cambiar tipo de entorno de ejecución > T4 GPU). Corre de cero con *Entorno de ejecución > Reiniciar y ejecutar todo*.

Qué hace, en orden:
1. Trae el repo y verifica las 40 imágenes del eval set de la tool.
2. **Prueba de humo**: carga CLIP y clasifica 3 imágenes, antes de bajar nada pesado.
3. Baja HAM10000 (~2.8 GB, verificado con md5) y arma validación y entrenamiento **por lesión**.
4. Compara candidatos: CLIP zero-shot (2 variantes), regresión logística sobre embeddings de CLIP y de ViT, y opcionalmente ViT + LoRA.
5. Elige con una regla fijada antes de correr, calibra el umbral y escribe `ver_config.json`.
6. Evalúa `ver()` (la función pública que usará el agente) sobre las 40 imágenes y mide la latencia.
7. Prueba qué pasa con fotos malas (la válvula `no_evaluable`).

**Regla de elección (fijada antes de correr):** mayor especificidad con recall de urgentes >= 0.85 en la partición de VALIDACIÓN; si empatan, menor latencia. Las 40 imágenes de evaluación no se usan para elegir ni para calibrar.

Los tiempos de cada paso son una estimación, no una medición.

In [18]:
# ---- Parámetros ----
RAMA = "camilo"          # rama que tiene la carpeta m4/ (cámbiala a "main" cuando esté fusionada)
ENTRENAR_LORA = True    # True agrega el candidato ViT + LoRA (unos minutos más en una T4)
RECALL_MIN = 0.85        # meta de recall de urgentes del equipo, usada para calibrar el umbral
SEMILLA = 42

import os
REPO = "/content/triage-dermatologico-ia"
if os.path.isdir(REPO):
    %cd /content/triage-dermatologico-ia
    !git fetch origin
    !git checkout {RAMA}
    !git pull origin {RAMA}
else:
    !git clone --branch {RAMA} https://github.com/Isa-Idarraga/triage-dermatologico-ia.git
    %cd /content/triage-dermatologico-ia
!git log -1 --oneline

/content/triage-dermatologico-ia
Already on 'camilo'
Your branch is up to date with 'origin/camilo'.
From https://github.com/Isa-Idarraga/triage-dermatologico-ia
 * branch            camilo     -> FETCH_HEAD
Already up to date.
0780fbf (HEAD -> camilo, origin/camilo) Avances M4 inicial Camilo


In [19]:
import platform
import numpy
import PIL
import sklearn
import torch
import transformers

hay_gpu = torch.cuda.is_available()
HARDWARE = torch.cuda.get_device_name(0) if hay_gpu else "CPU"
VERSIONES = {"python": platform.python_version(), "torch": torch.__version__,
             "transformers": transformers.__version__, "sklearn": sklearn.__version__,
             "numpy": numpy.__version__, "pillow": PIL.__version__}
print("Hardware:", HARDWARE)
print(VERSIONES)
if not hay_gpu:
    print("ADVERTENCIA: no hay GPU. Entorno de ejecución > Cambiar tipo de entorno > T4 GPU. "
          "Funciona igual pero lento, y la latencia que midas no es comparable.")

Hardware: Tesla T4
{'python': '3.13.15', 'torch': '2.11.0+cu130', 'transformers': '5.20.1', 'sklearn': '1.6.1', 'numpy': '2.1.3', 'pillow': '11.3.0'}


In [20]:
import io
import sys
import json
import time
import csv
import hashlib
import shutil
import zipfile
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFilter

PIPE = Path("m4/pipeline").resolve()
EVAL = PIPE / "eval_tool"
DATOS = Path("/content/ham10000")
sys.path.insert(0, str(PIPE))
import ham10000 as ham
import ver
import evaluacion_tool as ev

np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)

## 1 · Las 40 imágenes del eval set de la tool
Salen de la partición **reservada** de HAM10000: ninguna lesión de este conjunto se usa para entrenar ni para calibrar. Están en el repo; si faltan, se regeneran con el mismo script (determinista).

In [21]:
if not (EVAL / "manifiesto.csv").exists():
    print("Las imágenes no están en el repo: se descargan ahora (~11 MB).")
    !python m4/pipeline/seleccionar_imagenes.py

man = pd.read_csv(EVAL / "manifiesto.csv")
for r in man.itertuples():
    sha = hashlib.sha256((EVAL / r.archivo).read_bytes()).hexdigest()
    assert sha == r.sha256, f"{r.archivo} no coincide con su sha256"
print(len(man), "imágenes verificadas por sha256")
display(man.groupby(["esperado", "dx"]).size().rename("n").to_frame())

40 imágenes verificadas por sha256


n
esperado   dx      
no_urgente bkl    6
           df     3
           nv     8
           vasc   3
urgente    akiec  6
           bcc    6
           mel    8

## 2 · Prueba de humo (antes de bajar 2.8 GB)
Carga CLIP zero-shot con los valores por defecto y clasifica 3 imágenes: un melanoma, un carcinoma basocelular y un nevus. Si algo de la API de `transformers` falla, falla aquí y no 10 minutos después. La primera llamada incluye el arranque en frío de la GPU.

In [22]:
cfg_base = ver.cargar_config(PIPE / "no_existe.json")   # sin ver_config.json: CLIP zero-shot, umbral 0.5
t0 = time.time()
clf_humo = ver.construir_clasificador(cfg_base)
print(f"CLIP cargado en {time.time() - t0:.1f} s, dispositivo: {clf_humo.extractor.dispositivo}")
for dx in ("mel", "bcc", "nv"):
    r = man[man.dx == dx].iloc[0]
    out = ver.ver(EVAL / r.archivo, config=cfg_base, clasificador=clf_humo)
    print(f"{r.image_id} ({dx}, esperado {r.esperado}) -> {out['clase']}  score={out['score']}  {out['latencia_s']} s")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIP cargado en 4.4 s, dispositivo: cuda
ISIC_0025987 (mel, esperado urgente) -> urgente  score=0.8853  0.0413 s
ISIC_0025102 (bcc, esperado urgente) -> urgente  score=0.8304  0.0356 s
ISIC_0024876 (nv, esperado no_urgente) -> urgente  score=0.9295  0.0358 s


## 3 · Validación y entrenamiento (descarga ~2.8 GB)
Se baja HAM10000 completo, verificando el md5 que publica Dataverse. La partición se decide por **lesion_id** (1 956 lesiones tienen más de una foto): si fotos de la misma lesión cayeran a lados distintos, el modelo ya la habría visto y las métricas saldrían infladas. La celda lo comprueba con un `assert`.

In [23]:
t0 = time.time()
ham.descargar_imagenes(DATOS)
print(f"zips listos y verificados por md5 en {time.time() - t0:.0f} s")
fuente = ham.FuenteHAM.local(DATOS)
filas = ham.cargar_metadatos(DATOS)

val = [f for f in filas if f["particion"] == "validacion"]
ent = [f for f in filas if f["particion"] == "entrenamiento"]
res_ = [f for f in filas if f["particion"] == "reservado"]
les = [set(f["lesion_id"] for f in g) for g in (val, ent, res_)]
assert not (les[0] & les[1]) and not (les[0] & les[2]) and not (les[1] & les[2]), "hay lesiones en más de una partición"
for nombre, g in (("validación", val), ("entrenamiento", ent), ("reservado", res_)):
    print(f"{nombre:14s} {len(g):5d} imágenes, {sum(f['urgencia'] == 'urgente' for f in g):4d} urgentes")

  ya estaba: HAM10000_images_part_1.zip
  ya estaba: HAM10000_images_part_2.zip
zips listos y verificados por md5 en 10 s
validación      1469 imágenes,  292 urgentes
entrenamiento   6575 imágenes, 1253 urgentes
reservado       1971 imágenes,  409 urgentes


## 4 · Embeddings
Una sola lectura de cada imagen alimenta los dos extractores (CLIP y ViT). Se guardan en disco para no recalcular si se vuelve a correr la celda.

In [24]:
ext_clip = clf_humo.extractor            # reutiliza el CLIP ya cargado
ext_vit = ver.ExtractorViT(ver.MODELO_VIT)


def leer_pil(image_id):
    img = Image.open(io.BytesIO(fuente.leer(image_id)))
    img.load()
    return img.convert("RGB")


def embeddings(filas_, nombre_cache, tam_lote=64):
    ruta = DATOS / f"emb_{nombre_cache}.npz"
    ids = [f["image_id"] for f in filas_]
    if ruta.exists():
        z = np.load(ruta, allow_pickle=False)
        if list(z["ids"]) == ids:
            return z["clip"], z["vit"]
    salida_clip, salida_vit = [], []
    for i in range(0, len(ids), tam_lote):
        imgs = [leer_pil(j) for j in ids[i:i + tam_lote]]
        salida_clip.append(ext_clip.embeddings_imagen(imgs))
        salida_vit.append(ext_vit.embeddings_imagen(imgs))
    e_clip, e_vit = np.concatenate(salida_clip), np.concatenate(salida_vit)
    np.savez(ruta, ids=np.array(ids), clip=e_clip, vit=e_vit)
    return e_clip, e_vit


t0 = time.time()
E_val_clip, E_val_vit = embeddings(val, "validacion")
E_ent_clip, E_ent_vit = embeddings(ent, "entrenamiento")
imgs40 = [Image.open(EVAL / a).convert("RGB") for a in man.archivo]
E_40_clip = ext_clip.embeddings_imagen(imgs40)
E_40_vit = ext_vit.embeddings_imagen(imgs40)

y_val = np.array([f["urgencia"] == "urgente" for f in val]).astype(int)
y_ent = np.array([f["urgencia"] == "urgente" for f in ent]).astype(int)
y_40 = (man.esperado == "urgente").astype(int).to_numpy()
print(f"embeddings listos en {time.time() - t0:.0f} s | val {E_val_clip.shape} | ent {E_ent_clip.shape} | 40 {E_40_clip.shape}")

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     |  | 
--------------------+------------+--+-
pooler.dense.weight | UNEXPECTED |  | 
pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


embeddings listos en 1 s | val (1469, 512) | ent (6575, 512) | 40 (40, 512)


## 5 · Candidatos
- `clip_binario`: CLIP zero-shot con los dos prompts del reparto (melanoma vs nevus benigno).
- `clip_7clases`: CLIP zero-shot con un prompt por clase de HAM10000; el score es la probabilidad de las clases urgentes.
- `lr_clip` y `lr_vit`: regresión logística sobre los embeddings (la fuerza de regularización C se escoge por AUC en validación).

Los prompts están fijados en `ver.py` y no se ajustan mirando resultados.

In [25]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler


def ajustar_lr(E_ent, y_ent_, E_val, y_val_):
    esc = StandardScaler().fit(E_ent)
    mejor = None
    for C in (0.01, 0.1, 1.0):
        lr = LogisticRegression(C=C, class_weight="balanced", max_iter=2000, random_state=SEMILLA)
        lr.fit(esc.transform(E_ent), y_ent_)
        a = ev.auc(y_val_, lr.predict_proba(esc.transform(E_val))[:, 1])
        if mejor is None or a > mejor[0]:
            mejor = (a, C, lr)
    _, C, lr = mejor
    return C, dict(media=esc.mean_, escala=esc.scale_, coef=lr.coef_[0], sesgo=float(lr.intercept_[0]))


def puntuar_lr(pesos, E):
    z = ((E - pesos["media"]) / pesos["escala"]) @ pesos["coef"] + pesos["sesgo"]
    return 1.0 / (1.0 + np.exp(-np.clip(z, -60, 60)))


zs_bin = ver.ZeroShotCLIP(ext_clip, "binario")
zs_7 = ver.ZeroShotCLIP(ext_clip, "7clases")
C_clip, W_clip = ajustar_lr(E_ent_clip, y_ent, E_val_clip, y_val)
C_vit, W_vit = ajustar_lr(E_ent_vit, y_ent, E_val_vit, y_val)
print("C elegido: lr_clip =", C_clip, "| lr_vit =", C_vit)

ver.guardar_regresion(DATOS / "lr_clip.npz", **W_clip)
ver.guardar_regresion(DATOS / "lr_vit.npz", **W_vit)

# Objetos con la misma interfaz que usa ver() (puntuar(imagenes)) y scores sobre validación y las 40.
OBJ = {
    "clip_binario": zs_bin,
    "clip_7clases": zs_7,
    "lr_clip": ver.RegresionLogistica(ext_clip, DATOS / "lr_clip.npz"),
    "lr_vit": ver.RegresionLogistica(ext_vit, DATOS / "lr_vit.npz"),
}
INFO = {
    "clip_binario": {"modelo": ver.MODELO_CLIP, "pesos_tmp": None},
    "clip_7clases": {"modelo": ver.MODELO_CLIP, "pesos_tmp": None},
    "lr_clip": {"modelo": ver.MODELO_CLIP, "pesos_tmp": DATOS / "lr_clip.npz"},
    "lr_vit": {"modelo": ver.MODELO_VIT, "pesos_tmp": DATOS / "lr_vit.npz"},
}
S_val = {
    "clip_binario": zs_bin.puntuar_embeddings(E_val_clip),
    "clip_7clases": zs_7.puntuar_embeddings(E_val_clip),
    "lr_clip": puntuar_lr(W_clip, E_val_clip),
    "lr_vit": puntuar_lr(W_vit, E_val_vit),
}
S_40 = {
    "clip_binario": zs_bin.puntuar_embeddings(E_40_clip),
    "clip_7clases": zs_7.puntuar_embeddings(E_40_clip),
    "lr_clip": puntuar_lr(W_clip, E_40_clip),
    "lr_vit": puntuar_lr(W_vit, E_40_vit),
}

# Comprobación de integración: el objeto que usará ver() reproduce los scores calculados con los embeddings.
imgs_chk = [leer_pil(f["image_id"]) for f in val[:48]]
for nombre, obj in OBJ.items():
    dif = float(np.abs(obj.puntuar(imgs_chk) - S_val[nombre][:48]).max())
    print(f"{nombre:13s} diferencia máxima objeto vs embeddings: {dif:.5f}")
    assert dif < 1e-3, f"{nombre}: el objeto de ver() no reproduce el score de los embeddings"

C elegido: lr_clip = 0.01 | lr_vit = 0.01
clip_binario  diferencia máxima objeto vs embeddings: 0.00001
clip_7clases  diferencia máxima objeto vs embeddings: 0.00000
lr_clip       diferencia máxima objeto vs embeddings: 0.00000
lr_vit        diferencia máxima objeto vs embeddings: 0.00000


### 5b · ViT + LoRA (opcional, `ENTRENAR_LORA = True`)
Va más allá de lo visto en clase. Se entrena un ViT-B/16 con adaptador LoRA (r=8, sobre `query` y `value`) y la cabeza `classifier` guardada con el adaptador (`modules_to_save`): es la lección del incidente del `pooler` en M1. Se entrena con un subconjunto balanceado del entrenamiento (todos los urgentes y el mismo número de no urgentes). La celda comprueba que el adaptador recargado desde disco da los mismos scores que el modelo en memoria.

In [27]:
LORA_TMP = DATOS / "ver_lora"
if ENTRENAR_LORA:
    %pip install -q "peft>=0.11"
    !pip uninstall -y -q torchao
    import torch.nn.functional as F
    from peft import LoraConfig, get_peft_model
    from transformers import ViTForImageClassification

    dev = "cuda" if hay_gpu else "cpu"
    rng = np.random.default_rng(SEMILLA)
    urg_tr = [f for f in ent if f["urgencia"] == "urgente"]
    nou_tr = [f for f in ent if f["urgencia"] == "no_urgente"]
    nou_tr = [nou_tr[i] for i in rng.choice(len(nou_tr), size=len(urg_tr), replace=False)]
    sub = urg_tr + nou_tr
    y_lora = np.array([1] * len(urg_tr) + [0] * len(nou_tr))
    X = np.stack([np.asarray(leer_pil(f["image_id"]).resize((224, 224), Image.Resampling.BILINEAR)) for f in sub])
    print("entrenamiento LoRA:", X.shape, "| urgentes:", int(y_lora.sum()))

    base = ViTForImageClassification.from_pretrained(
        ver.MODELO_VIT, num_labels=2, id2label={0: "no_urgente", 1: "urgente"}, label2id={"no_urgente": 0, "urgente": 1})
    # Los nombres de las capas de atención cambian entre versiones de transformers
    # (query/value antes, q_proj/v_proj en la 5.x): se detectan en vez de fijarlos.
    capas = {n.split(".")[-1] for n, m in base.named_modules() if isinstance(m, torch.nn.Linear)}
    objetivo = next((par for par in (["query", "value"], ["q_proj", "v_proj"]) if set(par) <= capas), None)
    assert objetivo, f"No encuentro las capas de atención de ViT. Capas lineales: {sorted(capas)}"
    print("LoRA sobre:", objetivo)
    cfg_lora = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, target_modules=objetivo,
                          modules_to_save=["classifier"])
    modelo_lora = get_peft_model(base, cfg_lora).to(dev)
    modelo_lora.print_trainable_parameters()

    EPOCAS, LOTE = 4, 32
    opt = torch.optim.AdamW([p for p in modelo_lora.parameters() if p.requires_grad], lr=3e-4, weight_decay=0.01)
    escalador = torch.amp.GradScaler("cuda", enabled=hay_gpu)
    t0 = time.time()
    for ep in range(EPOCAS):
        modelo_lora.train()
        perm = rng.permutation(len(X))
        total = 0.0
        for i in range(0, len(perm), LOTE):
            idx = perm[i:i + LOTE]
            xb = torch.from_numpy(X[idx]).to(dev).permute(0, 3, 1, 2).float() / 255.0
            xb = (xb - 0.5) / 0.5
            if rng.random() < 0.5:
                xb = xb.flip(3)
            if rng.random() < 0.5:
                xb = xb.flip(2)
            yb = torch.from_numpy(y_lora[idx]).long().to(dev)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=hay_gpu):
                perdida = F.cross_entropy(modelo_lora(pixel_values=xb).logits.float(), yb)
            opt.zero_grad()
            escalador.scale(perdida).backward()
            escalador.step(opt)
            escalador.update()
            total += float(perdida) * len(idx)
        print(f"época {ep + 1}/{EPOCAS}  pérdida {total / len(perm):.4f}  ({time.time() - t0:.0f} s)")

    # La lección del pooler: lo que se recarga desde disco debe dar lo mismo que el modelo en memoria.
    modelo_lora.eval()
    imgs_chk2 = [leer_pil(f["image_id"]) for f in val[:32]]
    with torch.no_grad():
        xb = torch.from_numpy(np.stack([ver.preprocesar_vit(im) for im in imgs_chk2])).to(dev)
        s_mem = torch.softmax(modelo_lora(pixel_values=xb).logits.float(), dim=-1)[:, 1].cpu().numpy()
    modelo_lora.save_pretrained(str(LORA_TMP))
    del modelo_lora, base, X
    if hay_gpu:
        torch.cuda.empty_cache()
    clf_lora = ver.ViTLoRA(ver.MODELO_VIT, LORA_TMP)
    dif = float(np.abs(s_mem - clf_lora.puntuar(imgs_chk2)).max())
    print(f"diferencia máxima memoria vs disco: {dif:.6f}")
    assert dif < 1e-3, "el adaptador recargado no reproduce el modelo entrenado: revisa modules_to_save"

    def puntuar_filas(clf, filas_, tam_lote=64):
        partes = [clf.puntuar([leer_pil(f["image_id"]) for f in filas_[i:i + tam_lote]])
                  for i in range(0, len(filas_), tam_lote)]
        return np.concatenate(partes)

    OBJ["vit_lora"] = clf_lora
    INFO["vit_lora"] = {"modelo": ver.MODELO_VIT, "pesos_tmp": LORA_TMP}
    S_val["vit_lora"] = puntuar_filas(clf_lora, val)
    S_40["vit_lora"] = clf_lora.puntuar(imgs40)
else:
    print("ENTRENAR_LORA = False: se omite ViT + LoRA.")

entrenamiento LoRA: (2506, 224, 224, 3) | urgentes: 1253


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


LoRA sobre: ['q_proj', 'v_proj']
trainable params: 296,450 || all params: 86,096,644 || trainable%: 0.3443


/tmp/ipykernel_1872/2958046000.py:55: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  total += float(perdida) * len(idx)


época 1/4  pérdida 0.5507  (18 s)
época 2/4  pérdida 0.3955  (37 s)
época 3/4  pérdida 0.3527  (57 s)
época 4/4  pérdida 0.2979  (77 s)


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


diferencia máxima memoria vs disco: 0.000000


## 6 · Elegir con la regla fijada antes de correr
Para cada candidato se calibra el umbral en **validación**: el más alto (mayor especificidad) con recall de urgentes >= 0.85. Se elige la mayor especificidad (redondeada a 2 decimales); si empatan, la menor latencia. El AUC es solo informativo.

In [28]:
def latencia_ms(clf, imagenes, repeticiones=20):
    for im in imagenes[:3]:
        clf.puntuar([im])                               # calentamiento
    tiempos = []
    for im in imagenes[:repeticiones]:
        if hay_gpu:
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        clf.puntuar([im])
        if hay_gpu:
            torch.cuda.synchronize()
        tiempos.append((time.perf_counter() - t0) * 1000)
    return float(np.median(tiempos))


filas_tabla = []
for nombre, s in S_val.items():
    cal = ev.umbral_para_recall(y_val, s, RECALL_MIN)
    filas_tabla.append({
        "candidato": nombre, "auc_val": ev.auc(y_val, s), "umbral": cal["umbral"],
        "recall_val": cal["recall"], "especificidad_val": cal["especificidad"],
        "latencia_ms": latencia_ms(OBJ[nombre], imgs40),
    })
tabla = pd.DataFrame(filas_tabla)
tabla["esp_redondeada"] = tabla.especificidad_val.round(2)
tabla = tabla.sort_values(["esp_redondeada", "latencia_ms"], ascending=[False, True]).reset_index(drop=True)
ELEGIDO = tabla.loc[0, "candidato"]
display(tabla.drop(columns="esp_redondeada").round(4))
print("ELEGIDO:", ELEGIDO)

,candidato,auc_val,umbral,recall_val,especificidad_val,latencia_ms
0,lr_vit,0.9059,0.5310,0.8527,0.8267,51.3419
1,vit_lora,0.9176,0.6705,0.8527,0.8284,64.0776
2,lr_clip,0.9083,0.4646,0.8527,0.7961,46.4787
3,clip_7clases,0.5147,0.4167,0.8527,0.2048,42.4477
4,clip_binario,0.5094,0.3876,0.8527,0.1733,16.8496


ELEGIDO: lr_vit


### Informativo: los candidatos sobre las 40 imágenes
**No se usa para elegir.** Sirve para el README: cuánto rinde cada alternativa con su umbral calibrado en validación, incluidos los que se descartan. Con 20 urgentes y 20 no urgentes, cada imagen pesa 5 puntos de su clase.

In [29]:
filas_40 = []
for nombre, s in S_40.items():
    umbral = float(tabla.set_index("candidato").loc[nombre, "umbral"])
    clases = ["urgente" if v >= umbral else "no_urgente" for v in s]
    m = ev.resumen_tool(list(man.esperado), clases)
    filas_40.append({"candidato": nombre, "accuracy": m["accuracy"], "recall_urgente": m["recall_urgente"],
                     "especificidad": m["especificidad"], "auc_40": ev.auc(y_40, s)})
tabla_40 = pd.DataFrame(filas_40)
display(tabla_40.round(3))

,candidato,accuracy,recall_urgente,especificidad,auc_40
0,clip_binario,0.525,0.95,0.10,0.528
1,clip_7clases,0.525,0.95,0.10,0.568
2,lr_clip,0.650,0.95,0.35,0.758
3,lr_vit,0.500,0.60,0.40,0.668
4,vit_lora,0.625,0.70,0.55,0.708


## 7 · Escribir la configuración de `ver()`
El umbral sale de validación. La **nitidez mínima** (válvula de fotos desenfocadas) se calibra con las imágenes reales de validación: el percentil 0.5, o sea que por construcción se rechaza como mucho el 0.5 % de las fotos legítimas.

In [30]:
nitidez_val = np.array([ver.medir_calidad(leer_pil(f["image_id"]))[0] for f in val])
NITIDEZ_MIN = float(np.percentile(nitidez_val, 0.5))
print(f"nitidez de las fotos reales de validación: mediana {np.median(nitidez_val):.0f}, mínimo {nitidez_val.min():.1f}, "
      f"percentil 0.5 = {NITIDEZ_MIN:.1f}")

fila = tabla.set_index("candidato").loc[ELEGIDO]
if ELEGIDO in ("lr_clip", "lr_vit"):
    destino = f"ver_pesos_{ELEGIDO}.npz"
    shutil.copy(INFO[ELEGIDO]["pesos_tmp"], PIPE / destino)
    pesos_rel = destino
elif ELEGIDO == "vit_lora":
    if (PIPE / "ver_lora").exists():
        shutil.rmtree(PIPE / "ver_lora")
    shutil.copytree(INFO[ELEGIDO]["pesos_tmp"], PIPE / "ver_lora")
    pesos_rel = "ver_lora"
else:
    pesos_rel = None

cfg_final = {
    "backend": ELEGIDO,
    "modelo": INFO[ELEGIDO]["modelo"],
    "umbral": float(fila.umbral),
    "banda_incertidumbre": 0.0,
    "pesos": pesos_rel,
    "calidad": {"lado_minimo": 64, "desviacion_minima": 8.0, "nitidez_minima": NITIDEZ_MIN},
    "calibracion": {
        "regla": "mayor especificidad con recall de urgentes >= recall_min en validacion; empate, menor latencia",
        "recall_min": RECALL_MIN, "recall_val": float(fila.recall_val), "especificidad_val": float(fila.especificidad_val),
        "n_validacion": len(val), "semilla": SEMILLA, "hardware": HARDWARE, "versiones": VERSIONES,
    },
}
(PIPE / "ver_config.json").write_text(json.dumps(cfg_final, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in cfg_final.items() if k != "calibracion"}, ensure_ascii=False, indent=2))

nitidez de las fotos reales de validación: mediana 41, mínimo 9.4, percentil 0.5 = 11.3
{
  "backend": "lr_vit",
  "modelo": "google/vit-base-patch16-224-in21k",
  "umbral": 0.5310078860115258,
  "banda_incertidumbre": 0.0,
  "pesos": "ver_pesos_lr_vit.npz",
  "calidad": {
    "lado_minimo": 64,
    "desviacion_minima": 8.0,
    "nitidez_minima": 11.332866191864014
  }
}


## 8 · Evaluación final: `ver()` sobre las 40 imágenes
Se llama a la **función pública** `ver()` leyendo `ver_config.json`, igual que la llamará el agente. Incluye el filtro de calidad y la válvula. Las estrictas cuentan un `no_evaluable` sobre una lesión urgente como no detectada; `recall_con_escalamiento` lo cuenta como detectado, porque el sistema lo manda a revisión humana.

In [31]:
importlib.reload(ver)                  # relee ver_config.json; también vacía la caché del modelo
t0 = time.time()
ver.precargar()
CARGA_S = time.time() - t0
cfg = ver.cargar_config()
for a in man.archivo[:3]:
    ver.ver(EVAL / a)                  # calentamiento

filas_res = []
for r in man.itertuples():
    out = ver.ver(EVAL / r.archivo)
    filas_res.append({"image_id": r.image_id, "dx": r.dx, "esperado": r.esperado, "clase": out["clase"],
                      "score": out["score"], "motivo": out["motivo"], "latencia_s": out["latencia_s"]})
res = pd.DataFrame(filas_res)
M = ev.resumen_tool(res.esperado, res.clase)


def ic(par):
    return f"[{100 * par[0]:.0f}%, {100 * par[1]:.0f}%]"


print(f"Modelo: {cfg['backend']} ({cfg['modelo']}) | umbral {cfg['umbral']:.4f} | {HARDWARE}")
print(f"accuracy      {100 * M['accuracy']:5.1f}%  IC95 {ic(M['accuracy_ic95'])}   ({M['verdaderos_positivos'] + M['verdaderos_negativos']}/{M['n']})")
print(f"recall urgente {100 * M['recall_urgente']:5.1f}%  IC95 {ic(M['recall_urgente_ic95'])}   ({M['verdaderos_positivos']}/{M['n_urgentes']})")
print(f"especificidad {100 * M['especificidad']:5.1f}%  IC95 {ic(M['especificidad_ic95'])}   ({M['verdaderos_negativos']}/{M['n_no_urgentes']})")
print(f"no_evaluable: {M['no_evaluables_total']} | recall con escalamiento: {100 * M['recall_con_escalamiento']:.1f}%")
print(f"falsos negativos: {M['falsos_negativos']} | falsos positivos: {M['falsos_positivos']}")
print("Con 20 casos por clase, cada imagen pesa 5 puntos y los intervalos son anchos: léelo como orden de magnitud.")

display(res.assign(acierto=res.clase == res.esperado).groupby(["esperado", "dx"]).acierto.agg(aciertos="sum", total="count"))
errores = res[res.clase != res.esperado]
print("Errores y casos no evaluables:")
display(errores)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     |  | 
--------------------+------------+--+-
pooler.dense.weight | UNEXPECTED |  | 
pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modelo: lr_vit (google/vit-base-patch16-224-in21k) | umbral 0.5310 | Tesla T4
accuracy       50.0%  IC95 [35%, 65%]   (20/40)
recall urgente  60.0%  IC95 [39%, 78%]   (12/20)
especificidad  40.0%  IC95 [22%, 61%]   (8/20)
no_evaluable: 0 | recall con escalamiento: 60.0%
falsos negativos: 8 | falsos positivos: 12
Con 20 casos por clase, cada imagen pesa 5 puntos y los intervalos son anchos: léelo como orden de magnitud.


aciertos  total
esperado   dx                    
no_urgente bkl           2      6
           df            1      3
           nv            3      8
           vasc          2      3
urgente    akiec         4      6
           bcc           5      6
           mel           3      8

Errores y casos no evaluables:


,image_id,dx,esperado,clase,score,motivo,latencia_s
2,ISIC_0028558,akiec,urgente,no_urgente,0.2223,None,0.0362
4,ISIC_0029133,akiec,urgente,no_urgente,0.3205,None,0.0478
9,ISIC_0027858,bcc,urgente,no_urgente,0.4672,None,0.0371
12,ISIC_0025987,mel,urgente,no_urgente,0.4668,None,0.0341
14,ISIC_0032533,mel,urgente,no_urgente,0.3891,None,0.0492
15,ISIC_0032722,mel,urgente,no_urgente,0.2726,None,0.0321
17,ISIC_0033814,mel,urgente,no_urgente,0.2551,None,0.0309
18,ISIC_0033881,mel,urgente,no_urgente,0.3209,None,0.0320
20,ISIC_0025292,bkl,no_urgente,urgente,0.8640,None,0.0295
21,ISIC_0026085,bkl,no_urgente,urgente,0.5708,None,0.0299


### Latencia
Segundos por llamada a `ver()` (ya cargado el modelo y calentada la GPU), y el costo de cargarlo.

In [32]:
lat = res.latencia_s.to_numpy()
LAT = {"hardware": HARDWARE, "carga_modelo_s": round(CARGA_S, 2), "n_llamadas": len(lat),
       "mediana_s": round(float(np.median(lat)), 4), "media_s": round(float(lat.mean()), 4),
       "p95_s": round(float(np.percentile(lat, 95)), 4), "max_s": round(float(lat.max()), 4)}
print(LAT)

{'hardware': 'Tesla T4', 'carga_modelo_s': 0.45, 'n_llamadas': 40, 'mediana_s': 0.0298, 'media_s': 0.0322, 'p95_s': 0.0479, 'max_s': 0.0501}


## 9 · ¿Qué pasa con fotos malas?
Se degradan 10 de las 40 imágenes de cinco maneras y se mira qué responde `ver()`. Que la válvula atrape o no cada caso es un resultado, no un defecto de la prueba: sirve para el caso adversarial del eval set del sistema (`img_04`) y para la sección de límites.

In [33]:
def degradar(img, tipo):
    if tipo == "desenfoque fuerte":
        return img.filter(ImageFilter.GaussianBlur(15))
    if tipo == "muy pequeña (48 px)":
        return img.resize((48, 48))
    if tipo == "casi negra":
        return Image.eval(img, lambda v: int(v * 0.03))
    if tipo == "gris liso":
        return Image.new("RGB", img.size, (128, 128, 128))
    if tipo == "ruido":
        azar = np.random.default_rng(SEMILLA)
        return Image.fromarray(azar.integers(0, 255, (img.size[1], img.size[0], 3), dtype=np.uint8))
    raise ValueError(tipo)


TIPOS = ["desenfoque fuerte", "muy pequeña (48 px)", "casi negra", "gris liso", "ruido"]
muestra = list(man[man.esperado == "urgente"].archivo[:5]) + list(man[man.esperado == "no_urgente"].archivo[:5])
filas_deg = []
for tipo in TIPOS:
    for a in muestra:
        out = ver.ver(degradar(Image.open(EVAL / a).convert("RGB"), tipo))
        filas_deg.append({"tipo": tipo, "clase": out["clase"], "score": out["score"], "motivo": out["motivo"]})
deg = pd.DataFrame(filas_deg)
resumen_deg = deg.groupby("tipo").clase.value_counts().unstack(fill_value=0).reindex(TIPOS)
display(resumen_deg)
print("Cada fila suma 10 fotos. Las que NO se marcaron no_evaluable fueron clasificadas igual, con su score:")
print("esa es la limitación a documentar (la válvula detecta mala calidad, no que la foto no sea piel).")

clase,no_evaluable,urgente
tipo,,
desenfoque fuerte,10,0
muy pequeña (48 px),10,0
casi negra,10,0
gris liso,10,0
ruido,0,10


Cada fila suma 10 fotos. Las que NO se marcaron no_evaluable fueron clasificadas igual, con su score:
esa es la limitación a documentar (la válvula detecta mala calidad, no que la foto no sea piel).


## 10 · Guardar resultados y texto para el README

In [34]:
res.to_csv(EVAL / "resultados_tool.csv", index=False)
resumen = {
    "modelo": {k: cfg[k] for k in ("backend", "modelo", "umbral", "pesos")},
    "metricas_40": {k: (list(v) if isinstance(v, tuple) else v) for k, v in M.items()},
    "latencia": LAT,
    "candidatos_validacion": json.loads(tabla.drop(columns="esp_redondeada").to_json(orient="records")),
    "candidatos_40_informativo": json.loads(tabla_40.to_json(orient="records")),
    "fotos_degradadas": json.loads(resumen_deg.to_json()),
    "versiones": VERSIONES,
    "semilla": SEMILLA,
}
(EVAL / "resultados_tool.json").write_text(json.dumps(resumen, ensure_ascii=False, indent=2), encoding="utf-8")

print("Números para la sección 'Patrón y modelo' del README:\n")
print(f"- Modelo: {cfg['backend']} ({cfg['modelo']}), umbral {cfg['umbral']:.3f} calibrado en validación (recall >= {RECALL_MIN}).")
print(f"- Eval set de la tool: {M['n']} imágenes de HAM10000 ({M['n_urgentes']} urgentes, {M['n_no_urgentes']} no urgentes), partición reservada.")
print(f"- Accuracy {100 * M['accuracy']:.0f}% {ic(M['accuracy_ic95'])}; recall de urgentes {100 * M['recall_urgente']:.0f}% "
      f"{ic(M['recall_urgente_ic95'])}; especificidad {100 * M['especificidad']:.0f}% {ic(M['especificidad_ic95'])}.")
print(f"- Latencia: mediana {LAT['mediana_s']} s por llamada en {HARDWARE} (cargar el modelo: {LAT['carga_modelo_s']} s).")
alternativas = "; ".join(f"{r.candidato}: accuracy {100 * r.accuracy:.0f}%, recall {100 * r.recall_urgente:.0f}%"
                         for r in tabla_40.itertuples())
print("- Candidatos sobre las mismas 40 (umbral calibrado en validación): " + alternativas + ".")

Números para la sección 'Patrón y modelo' del README:

- Modelo: lr_vit (google/vit-base-patch16-224-in21k), umbral 0.531 calibrado en validación (recall >= 0.85).
- Eval set de la tool: 40 imágenes de HAM10000 (20 urgentes, 20 no urgentes), partición reservada.
- Accuracy 50% [35%, 65%]; recall de urgentes 60% [39%, 78%]; especificidad 40% [22%, 61%].
- Latencia: mediana 0.0298 s por llamada en Tesla T4 (cargar el modelo: 0.45 s).
- Candidatos sobre las mismas 40 (umbral calibrado en validación): clip_binario: accuracy 52%, recall 95%; clip_7clases: accuracy 52%, recall 95%; lr_clip: accuracy 65%, recall 95%; lr_vit: accuracy 50%, recall 60%; vit_lora: accuracy 62%, recall 70%.


## 11 · Llevarse los resultados al repo
Descarga un zip con la configuración, los pesos y los resultados. Descomprímelo **en la raíz del repo** (las rutas dentro del zip ya empiezan por `m4/pipeline/...`) y haz commit.

In [35]:
from google.colab import files

destino_zip = Path("/content/m4_resultados.zip")
raiz = PIPE.parent.parent
candidatos_zip = [PIPE / "ver_config.json", EVAL / "resultados_tool.csv", EVAL / "resultados_tool.json"]
candidatos_zip += list(PIPE.glob("ver_pesos_*.npz"))
if (PIPE / "ver_lora").exists():
    candidatos_zip += list((PIPE / "ver_lora").rglob("*"))
with zipfile.ZipFile(destino_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for p in candidatos_zip:
        if p.is_file():
            z.write(p, p.relative_to(raiz))
            print("en el zip:", p.relative_to(raiz))
files.download(str(destino_zip))

en el zip: m4/pipeline/ver_config.json
en el zip: m4/pipeline/eval_tool/resultados_tool.csv
en el zip: m4/pipeline/eval_tool/resultados_tool.json
en el zip: m4/pipeline/ver_pesos_lr_vit.npz


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>